In [2]:
pip install dash dash-bootstrap-components pandas plotly

Note: you may need to restart the kernel to use updated packages.


In [3]:
import dash
import dash_bootstrap_components as dbc
from dash import html, dcc, Input, Output
import plotly.express as px
import pandas as pd

In [4]:
df = pd.read_csv('SalesDataset.csv')

In [5]:
print(df.head())

  Order ID  Amount  Profit  Quantity     Category      Sub-Category  \
0  B-26776    9726    1275         5  Electronics  Electronic Games   
1  B-26776    9726    1275         5  Electronics  Electronic Games   
2  B-26776    9726    1275         5  Electronics  Electronic Games   
3  B-26776    4975    1330        14  Electronics          Printers   
4  B-26776    4975    1330        14  Electronics          Printers   

  PaymentMode  Order Date   CustomerName     State     City Year-Month  
0         UPI  2023-06-27  David Padilla   Florida    Miami    2023-06  
1         UPI  2024-12-27  Connor Morgan  Illinois  Chicago    2024-12  
2         UPI  2021-07-25   Robert Stone  New York  Buffalo    2021-07  
3         UPI  2023-06-27  David Padilla   Florida    Miami    2023-06  
4         UPI  2024-12-27  Connor Morgan  Illinois  Chicago    2024-12  


In [6]:
df['Order Date'] = pd.to_datetime(df['Order Date'])

In [7]:
df['Year-Month'] = df['Order Date'].dt.strftime('%Y-%m')
monthly_sales = df.groupby('Year-Month')['Amount'].sum().reset_index()

In [8]:
scatter_fig = px.scatter(df, x='Amount', y='Profit', color='Category', hover_data=['CustomerName'])

In [9]:
line_fig = px.line(monthly_sales, x='Year-Month', y='Amount', title='Monthly Sales Trend')

In [10]:
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Layout of the dashboard
app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H2("Sales Dashboard Overview"), width=12)
    ], className='my-2'),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=line_fig), width=12)
    ], className='my-2'),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=scatter_fig), width=12)
    ], className='my-2'),
    dbc.Row([
        dbc.Col(html.Div("Additional KPI or controls can go here..."), width=12)
    ], className='my-2')
], fluid=True)

In [11]:
if __name__ == '__main__':
    app.run(debug=True)

In [15]:
import dash
from dash import html, dcc

# Initialize the Dash app
app = dash.Dash(__name__)

# Define the app's layout using HTML and Core Components
app.layout = html.Div([
    # Title section with HTML components
    html.Div([
        html.H1("Sales Dashboard", style={'textAlign': 'center'}),
        html.P("An interactive dashboard built with Dash.", style={'textAlign': 'center'})
    ], style={'padding': '20px', 'backgroundColor': '#e8f4f8'}),

    # First section: Interactive Graph using dcc.Graph
    html.Div([
        dcc.Graph(
            id='sales-trend',
            figure={
                'data': [
                    {'x': ['2025-01', '2025-02', '2025-03', '2025-04'],
                     'y': [2500, 3000, 2800, 3200],
                     'type': 'line',
                     'name': 'Total Sales'}
                ],
                'layout': {
                    'title': 'Monthly Sales Trend'
                }
            }
        )
    ], style={'padding': '20px'}),

    # Second section: Dropdown for filtering example
    html.Div([
        html.Label("Select Product Category:"),
        dcc.Dropdown(
            id='product-category',
            options=[
                {'label': 'Electronics', 'value': 'Electronics'},
                {'label': 'Apparel', 'value': 'Apparel'},
                {'label': 'Home & Kitchen', 'value': 'Home & Kitchen'}
            ],
            value='Electronics'
        )
    ], style={'width': '50%', 'padding': '20px', 'margin': 'auto'})
])

# Run the Dash app
if __name__ == "__main__":
    app.run(debug=True)

In [17]:
df['Order_date'] = pd.to_datetime(df['Order Date'])
df['Year-Month'] = df['Order Date'].dt.strftime('%Y-%m')

# Initialize the Dash app with a Bootstrap theme for improved aesthetics
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Define the layout of the app using HTML and Core Components
app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("Interactive Sales Dashboard", style={'textAlign': 'center'}), width=12)
    ], className='my-3'),

    dbc.Row([
        dbc.Col([
            html.Label("Select Product Category:"),
            dcc.Dropdown(
                id='category-dropdown',
                options=[{'label': cat, 'value': cat} for cat in sorted(df['Category'].unique())],
                value=sorted(df['Category'].unique())[0],
                clearable=False
            )
        ], width=4),

        dbc.Col([
            html.Label("Minimum Sales Amount:"),
            dcc.Input(
                id='min-sales-input',
                type='number',
                placeholder='Enter minimum sales value',
                value=0
            )
        ], width=4)
    ], className='my-3'),

    dbc.Row([
        dbc.Col(
            dcc.Graph(id='line-chart'),
            width=12
        )
    ], className='my-3')
], fluid=True)

# Define the callback to update the line chart based on user inputs
@app.callback(
    Output('line-chart', 'figure'),
    Input('category-dropdown', 'value'),
    Input('min-sales-input', 'value')
)
def update_line_chart(selected_category, min_sales):
    """
    When the user selects a category and/or updates the minimum sales amount,
    this function filters the data accordingly, groups the sales by Year-Month,
    and creates an updated line chart.
    """
    # Filter the dataset by the selected category
    filtered_df = df[df['Category'] == selected_category]

    # Group data by Year-Month and sum the Amount
    grouped = filtered_df.groupby('Year-Month')['Amount'].sum().reset_index()

    # Apply minimum sales filtering: show only months above the specified threshold
    grouped = grouped[grouped['Amount'] >= min_sales]

    # Create a Plotly Express line chart
    fig = px.line(grouped, x='Year-Month', y='Amount',
                  title=f"Monthly Sales Trend for {selected_category}",
                  labels={'Amount': 'Total Sales Amount', 'Year-Month': 'Month'})

    # Rotate x-axis labels for clarity
    fig.update_layout(xaxis_tickangle=-45)
    return fig

# Run the Dash app
if __name__ == '__main__':
    app.run(debug=True)